In [1]:
import pandas as pd
import numpy as np

credit = pd.read_csv("data/german_credit_data.csv")
credit.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,NaN,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad


## Czyszczenie nazw kolumn

Czyszczenie *nazw* **kolumn** z wykorzystaniem pakietu `pyjanitor`.

In [2]:
from janitor import clean_names

credit = credit.clean_names()
credit.head()

,age,sex,job,housing,saving_accounts,checking_account,credit_amount,duration,purpose,risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,NaN,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad


## Braki danych

In [3]:
credit["checking_account"].value_counts()

checking_account
little      274
moderate    269
rich         63
Name: count, dtype: int64

In [4]:
check_acc_mode = credit["checking_account"].mode()[0]

In [5]:
credit["checking_account"] = credit["checking_account"].fillna(check_acc_mode)
credit["checking_account"].value_counts()

checking_account
little      668
moderate    269
rich         63
Name: count, dtype: int64

In [6]:
# ! pip install scikit-learn

In [7]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="most_frequent")
credit[["saving_accounts"]] = imputer.fit_transform(credit[["saving_accounts"]])

In [8]:
imputer.feature_names_in_

array(['saving_accounts'], dtype=object)

# Inżynieria cech

In [9]:
credit["installment_rate"] = credit["credit_amount"] / credit["duration"]
credit["age_groups"] = pd.cut(credit["age"], 
                              bins = [0, 30, 40, 60, np.inf], 
                              labels = ["<30", "31-40", "41-60", "60+"])

In [10]:
from sklearn.preprocessing import OrdinalEncoder

account_order = [["little", "moderate", "rich"]]

ordinal_encoder = OrdinalEncoder(categories=account_order)

credit["checking_account_encoded"] = ordinal_encoder.fit_transform(
    credit[["checking_account"]]
)

In [13]:
credit["job_avg_credit_amount"] = (
    credit.groupby("job")["credit_amount"]
    .transform("mean")
)

# One hot encoding

In [5]:
categorical_cols = ["sex", "job", "housing", "saving_accounts", "checking_account", "purpose", "age_groups"]
credit_ohe = pd.get_dummies(credit, columns=categorical_cols, dtype=int)

In [6]:
from sklearn.preprocessing import OneHotEncoder

ohe_encoder = OneHotEncoder()
credit_ohe_encoder = ohe_encoder.fit_transform(credit[categorical_cols])
credit_ohe_encoder_df = pd.DataFrame(credit_ohe_encoder.toarray(), 
                                     columns=ohe_encoder.get_feature_names_out(categorical_cols))

In [7]:
credit_ohe_encoder_df.columns

Index(['sex_female', 'sex_male', 'job_0', 'job_1', 'job_2', 'job_3', 'housing_free', 'housing_own', 'housing_rent',
       'saving_accounts_little', 'saving_accounts_moderate', 'saving_accounts_quite rich', 'saving_accounts_rich',
       'saving_accounts_nan', 'checking_account_little', 'checking_account_moderate', 'checking_account_rich',
       'checking_account_nan', 'purpose_business', 'purpose_car', 'purpose_domestic appliances', 'purpose_education',
       'purpose_furniture/equipment', 'purpose_radio/TV', 'purpose_repairs', 'purpose_vacation/others', 'age_groups_31-40',
       'age_groups_41-60', 'age_groups_60+', 'age_groups_<30'],
      dtype='str')

# Normalizacja cech

In [9]:
numeric_cols = ["age", "credit_amount", "duration", "installment_rate"]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
credit_scaled = scaler.fit_transform(credit[numeric_cols])
credit_scaled_df = pd.DataFrame(credit_scaled, columns=numeric_cols)

# Kodowanie etykiet

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le_risk = le.fit_transform(credit["risk"])

# Finalny zbiór danych

In [13]:
credit_final = pd.concat([credit_scaled_df, credit_ohe_encoder_df, pd.Series(le_risk, name="risk")], axis=1)

credit_final = credit_final.clean_names()

credit_final.to_csv("data/credit_final.csv", index=False)